# KV-кэш руками: тот же результат без повторной проекции прошлого

Вопрос опыта: изменится ли причинное внимание, если ключи и значения прошлых позиций не вычислять заново, а сохранить? Мы сравним оба пути на каждом шаге, посчитаем устранённые проекции и намеренно внесём ошибку границы кэша.

In [ ]:
import math
import platform

print('python:', platform.python_version())

def dot(left, right):
    return sum(a * b for a, b in zip(left, right))

def matvec(matrix, vector):
    return [dot(row, vector) for row in matrix]

def stable_softmax(values):
    maximum = max(values)
    weights = [math.exp(value - maximum) for value in values]
    total = sum(weights)
    return [weight / total for weight in weights]

def weighted_sum(weights, vectors):
    return [sum(weight * vector[j] for weight, vector in zip(weights, vectors))
            for j in range(len(vectors[0]))]

def assert_vector_close(actual, expected, tolerance=1e-12):
    assert len(actual) == len(expected)
    assert all(abs(a - b) <= tolerance for a, b in zip(actual, expected)), (actual, expected)

# Это состояния после добавления позиционной информации.
states = [
    [1.0, 0.0, 0.5],
    [0.0, 1.0, -0.5],
    [0.5, 0.5, 1.0],
    [-0.5, 1.0, 0.25],
    [1.0, -0.5, 0.0],
]
W_Q = [[1.0, 0.2, 0.0], [0.0, 0.8, -0.1], [0.3, 0.0, 0.7]]
W_K = [[0.9, 0.0, 0.1], [0.1, 1.0, 0.0], [0.0, -0.2, 0.8]]
W_V = [[1.0, 0.0, -0.2], [0.0, 0.5, 0.5], [0.2, -0.1, 0.9]]

## 1. Эталон без кэша

Для очередной позиции эталон заново строит ключи и значения всего префикса. Причинная маска здесь выражена самим выбором префикса: будущие состояния функции не передаются.

In [ ]:
def attention_last_without_cache(prefix):
    query = matvec(W_Q, prefix[-1])
    keys = [matvec(W_K, state) for state in prefix]
    values = [matvec(W_V, state) for state in prefix]
    scores = [dot(query, key) / math.sqrt(len(query)) for key in keys]
    return weighted_sum(stable_softmax(scores), values)

reference_outputs = [attention_last_without_cache(states[:end])
                     for end in range(1, len(states) + 1)]
for position, output in enumerate(reference_outputs):
    print(position, [round(value, 6) for value in output])

## 2. Пошаговый путь с кэшем

На шаге создаются только текущие query, key и value. Новый key/value дописывается до вычисления внимания, поэтому текущая позиция видит саму себя. Прошлые элементы кэша не меняются.

In [ ]:
cached_keys = []
cached_values = []
cached_outputs = []

for state in states:
    query = matvec(W_Q, state)
    cached_keys.append(matvec(W_K, state))
    cached_values.append(matvec(W_V, state))
    scores = [dot(query, key) / math.sqrt(len(query)) for key in cached_keys]
    output = weighted_sum(stable_softmax(scores), cached_values)
    cached_outputs.append(output)

for actual, expected in zip(cached_outputs, reference_outputs):
    assert_vector_close(actual, expected)

assert len(cached_keys) == len(states)
assert len(cached_values) == len(states)
print('Все', len(states), 'пошаговых выходов совпали с полным пересчётом.')

## 3. Что именно сэкономлено и что заняло память

Для последовательности длины T эталон T раз строит query и на каждом шаге заново строит K/V всего префикса. Кэш строит ровно по одной тройке Q/K/V на позицию. Само умножение query на прошлые keys остаётся: кэш устраняет повторную проекцию и повторный проход прошлых токенов, а не зависимость внимания от длины контекста.

In [ ]:
T = len(states)
projections_without_cache = T + 2 * sum(range(1, T + 1))
projections_with_cache = 3 * T
assert projections_without_cache == 35
assert projections_with_cache == 15
print('проекций без кэша:', projections_without_cache)
print('проекций с кэшем:  ', projections_with_cache)

def kv_bytes(layers, tokens, kv_heads, head_dim, bytes_per_element):
    return 2 * layers * tokens * kv_heads * head_dim * bytes_per_element

layers, tokens, head_dim, element_bytes = 32, 8192, 128, 2
mha = kv_bytes(layers, tokens, 32, head_dim, element_bytes)
gqa = kv_bytes(layers, tokens, 8, head_dim, element_bytes)
assert mha == 4 * gqa
print('MHA KV GiB:', round(mha / 2**30, 3))
print('GQA KV GiB:', round(gqa / 2**30, 3))

## 4. Намеренная ошибка: текущая позиция не попала в кэш вовремя

Если вычислить внимание до добавления текущих K/V, позиция перестаёт видеть саму себя. Формы остаются правдоподобными, программа не падает, но численный контракт нарушается.

In [ ]:
bad_keys, bad_values, bad_outputs = [], [], []
for state in states:
    query = matvec(W_Q, state)
    if bad_keys:
        scores = [dot(query, key) / math.sqrt(len(query)) for key in bad_keys]
        bad_outputs.append(weighted_sum(stable_softmax(scores), bad_values))
    bad_keys.append(matvec(W_K, state))
    bad_values.append(matvec(W_V, state))

differences = [max(abs(a - b) for a, b in zip(actual, expected))
               for actual, expected in zip(bad_outputs, reference_outputs[1:])]
assert max(differences) > 1e-3
print('ошибка обнаружена, максимальное расхождение:', round(max(differences), 6))

## 5. Переход к промышленной системе

Переменные cached_keys и cached_values соответствуют тензорам кэша слоя; индекс списка — логической позиции. В блочном распределителе vLLM логическая позиция сначала переводится таблицей блоков в физический адрес. Формула памяти остаётся той же, но к полезным данным добавляются незаполненные места блоков и служебные таблицы.

**Задание.** Добавьте два слоя, пакет из двух последовательностей разной длины и счётчик занятых слотов. Затем намеренно перепутайте физические блоки запросов и докажите ошибку сравнением с эталоном.